# Testing CHRONOSv2 on PEMS-BAY and other graph datasets

In [2]:
import numpy as np
import torch
import time
import pandas as pd
from tqdm import tqdm
from chronos import Chronos2Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- Configuration ---
# Ensure this points to your .dat file
FILE_PATH = '../data/PEMS-BAY/data.dat' 
# Total shape from your original post: (52116, 325, 3)
FULL_SHAPE = (52116, 325, 3) 
DTYPE = 'float32'

CONTEXT_LEN = 288
HORIZONS = [3, 6, 12]
TEST_RATIO = 0.20
NUM_RUNS = 3

def calculate_metrics(y_true, y_pred):
    """Calculates metrics across all nodes and time steps in the window."""
    mae = np.mean(np.abs(y_true - y_pred))
    mse = np.mean((y_true - y_pred)**2)
    rmse = np.sqrt(mse)
    # MAPE calculation with a floor to avoid division by zero
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1.0, None))) * 100
    return mae, rmse, mse, mape

def run_experiment():
    # 1. Load the data using memmap
    # We only need feature 0 (Speed), so we will slice it carefully
    data_mm = np.memmap(FILE_PATH, dtype=DTYPE, mode='r', shape=FULL_SHAPE)
    
    total_steps = FULL_SHAPE[0]
    num_nodes = FULL_SHAPE[1]
    test_start = int(total_steps * (1 - TEST_RATIO))
    max_h = max(HORIZONS)
    
    # Define evaluation indices (sliding by max_h to cover the 20% test set)
    indices = np.arange(test_start, total_steps - max_h, max_h)

    print(f"Loading Chronos-2-Small on {'cuda' if torch.cuda.is_available() else 'cpu'}...")
    pipeline = Chronos2Pipeline.from_pretrained(
        "amazon/chronos-2",
        device_map="cuda" if torch.cuda.is_available() else "cpu",
        dtype=torch.bfloat16,
    )

    all_run_results = []

    for run in range(NUM_RUNS):
        print(f"\n>>> Run {run+1}/{NUM_RUNS}")
        start_time = time.time()
        
        h_metrics = {h: {'mae': [], 'rmse': [], 'mse': [], 'mape': []} for h in HORIZONS}

        for idx in tqdm(indices, desc="Batch Predicting Nodes"):
            # 2. Prepare Context
            # Slice: [Time, Nodes, Feature_0] -> then Transpose to [Nodes, Time]
            # Shape becomes (325, 12)
            context_slice = data_mm[idx - CONTEXT_LEN : idx, :, 0].T
            
            # Chronos-2 expects (Series, Variates, Time)
            # We treat the whole Bay Area as 1 Series with 325 Variates
            context = torch.tensor(context_slice, dtype=torch.float32).unsqueeze(0)
            
            # Ground Truth: [Nodes, Max_H]
            ground_truth_full = data_mm[idx : idx + max_h, :, 0].T
            
            with torch.no_grad():
                # One model pass predicts all 325 nodes simultaneously
                forecast = pipeline.predict(context, prediction_length=max_h)
            
            # forecast shape: [1, 325, max_h, 3] -> (Series, Variates, Time, Quantiles)
            # Index 1 is the 0.5 quantile (Median)
            prediction_tensor = forecast[0]

            if prediction_tensor.ndim == 4:
                # Shape: [Series, Variates, Time, Quantiles]
                prediction = prediction_tensor[0, :, :, 1].cpu().numpy()
            else:
                # Shape: [Variates, Time, Quantiles]
                prediction = prediction_tensor[:, :, 1].cpu().numpy()

            for h in HORIZONS:
                m_mae, m_rmse, m_mse, m_mape = calculate_metrics(
                    ground_truth_full[:, :h], prediction[:, :h]
                )
                h_metrics[h]['mae'].append(m_mae)
                h_metrics[h]['rmse'].append(m_rmse)
                h_metrics[h]['mse'].append(m_mse)
                h_metrics[h]['mape'].append(m_mape)

        run_duration = time.time() - start_time
        
        # Aggregate results for this run
        res = {"time_sec": run_duration}
        for h in HORIZONS:
            for m in ['mae', 'rmse', 'mse', 'mape']:
                res[f"h{h}_{m}"] = np.mean(h_metrics[h][m])
        all_run_results.append(res)

    # --- Final Output Formatting ---
    df = pd.DataFrame(all_run_results)
    
    # Calculate averages and std across the 3 runs for the CSV row
    results_row = {"method": "chronos-2-multivariate"}
    for h in HORIZONS:
        for m in ['mae', 'rmse', 'mape']:
            results_row[f"test_{m}_h{h}_mean"] = df[f"h{h}_{m}"].mean()
            results_row[f"test_{m}_h{h}_std"] = df[f"h{h}_{m}"].std()

    # Calculate global averages (average of h3, h6, h12)
    avg_mae = df[[f"h{h}_mae" for h in HORIZONS]].mean(axis=1)
    avg_rmse = df[[f"h{h}_rmse" for h in HORIZONS]].mean(axis=1)

    print("\n" + "="*40)
    print(f'"chronos-2-pems": {{')
    print(f'    "test_mae": "{avg_mae.mean():.4f} +- {avg_mae.std():.4f}",')
    print(f'    "test_rmse": "{avg_rmse.mean():.4f} +- {avg_rmse.std():.4f}",')
    print(f'    "time_sec": "{df["time_sec"].mean():.2f}"')
    print(f'}}')
    print("="*40)
    
    # Save the row to CSV as requested
    pd.DataFrame([results_row]).to_csv("chronos_pems_benchmark.csv", index=False)
    print("Results saved to chronos_pems_benchmark.csv")

if __name__ == "__main__":
    run_experiment()

Loading Chronos-2-Small on cuda...

>>> Run 1/3


Batch Predicting Nodes:   1%|          | 5/868 [00:04<13:47,  1.04it/s]


KeyboardInterrupt: 